# Real MRI smoke-test results

Inspect saved TotalSegmentator MRI and UniverSeg predictions on two CrossPan development cases.
This notebook **does not download files or run models**. Run the model commands first.

This is a technical smoke test, not a fair ranking: TotalSegmentator is automatic, while
UniverSeg receives labeled slices from a separate same-sequence support patient. These are
two convenience cases from different cohorts; pretraining overlap is unverified. The T1
case has only nine slices at 23.2 mm spacing. No sequence-generalization claim follows.

Dice uses every voxel in each original volume. The displayed slice is selected from the
reference only for visualization; it is not used to crop model inputs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from ml_biomed_project.metrics import dice_score

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
run = project / "results" / "first-mri-smoke-test"
cases = [("T1", "EMC014_T1"), ("T2", "CAD_T2_110")]
rows = []
figures = []

In [ ]:
for sequence, case in cases:
    folder = project / "data" / "crosspan" / sequence / "val"
    image = nib.load(folder / "images" / f"{case}_0000.nii.gz")
    reference = nib.load(folder / "labels" / f"{case}.nii.gz")
    if image.shape != reference.shape or not np.allclose(
        image.affine, reference.affine, rtol=0, atol=1e-5
    ):
        raise ValueError(f"Reference grid mismatch: {case}")
    data = image.get_fdata(dtype=np.float32)
    truth = np.asarray(reference.dataobj)
    if not np.all(np.isfinite(data)) or not np.all((truth == 0) | (truth == 1)):
        raise ValueError(f"Invalid image values or reference labels: {case}")
    if image.header.get_xyzt_units()[0] != reference.header.get_xyzt_units()[0]:
        raise ValueError(f"Spatial units mismatch: {case}")
    slice_index = int(np.argmax(np.count_nonzero(truth, axis=(0, 1))))
    spacing = image.header.get_zooms()
    limits = np.percentile(data, [1, 99])
    predictions = {"Reference": truth}
    for model, filename in (
        ("totalsegmentator", "multilabel.nii.gz"),
        ("universeg", "pancreas.nii.gz"),
    ):
        path = run / model / case / filename
        if not path.exists():
            rows.append({"case": case, "model": model, "status": "missing output"})
            continue
        volume = nib.load(path)
        if volume.shape != image.shape or not np.allclose(
            volume.affine, image.affine, rtol=0, atol=1e-5
        ):
            raise ValueError(f"Prediction grid mismatch: {model} {case}")
        if volume.header.get_xyzt_units()[0] != image.header.get_xyzt_units()[0]:
            raise ValueError(f"Prediction spatial units mismatch: {model} {case}")
        values = np.asarray(volume.dataobj)
        if not np.all(np.isfinite(values)) or not np.all(values == np.floor(values)):
            raise ValueError(f"Invalid prediction labels: {model} {case}")
        if model == "totalsegmentator":
            # Pancreas is class 7 in the pinned total_mr multilabel mapping.
            prediction = values == 7
        else:
            if not np.all((values == 0) | (values == 1)):
                raise ValueError(f"UniverSeg prediction must be binary: {case}")
            prediction = values.astype(bool)
        score = dice_score(prediction, truth)
        rows.append(
            {
                "case": case,
                "model": model,
                "status": "evaluated",
                "dice": score,
                "prediction_voxels": int(prediction.sum()),
                "reference_voxels": int(truth.sum()),
            }
        )
        predictions[model] = prediction

    fig, axes = plt.subplots(
        1, len(predictions), figsize=(5 * len(predictions), 4), layout="constrained"
    )
    for ax, (name, mask) in zip(np.atleast_1d(axes), predictions.items(), strict=True):
        ax.imshow(
            data[:, :, slice_index].T,
            cmap="gray",
            origin="lower",
            aspect=float(spacing[1] / spacing[0]),
            vmin=limits[0],
            vmax=limits[1],
        )
        reference_slice = truth[:, :, slice_index].T
        mask_slice = mask[:, :, slice_index].T
        if np.any(reference_slice):
            ax.contour(reference_slice, levels=[0.5], colors=["cyan"], linewidths=1)
        if name != "Reference" and np.any(mask_slice):
            ax.contour(mask_slice, levels=[0.5], colors=["orange"], linewidths=1)
        ax.set(title=name, xlabel="Array axis 0 (voxels)", ylabel="Array axis 1 (voxels)")
    fig.suptitle(f"{case} — slice {slice_index}; cyan: reference, orange: prediction")
    figures.append((case, fig))
    plt.show()

scores = pd.DataFrame(rows)
print(scores.to_string(index=False))

The table reports per-case whole-volume Dice, not the score of the displayed slice.
Missing outputs remain explicit; inspect each attempt's local log before interpreting them.
An empty prediction against a nonempty reference scores zero. Two empty masks produce NaN,
which must remain visible rather than being silently excluded.

Optional: run the following cell to save this table and the overlays locally. These results
are ignored by Git; committed notebooks should have all outputs cleared.

In [ ]:
run.mkdir(parents=True, exist_ok=True)
scores.to_csv(run / "scores.csv", index=False)
for case, figure in figures:
    figure.savefig(run / f"{case}-overlay.png", dpi=140)
print(f"Saved results under {run}")